# 06 · Clustering and label transfer

Following the tutorial notebook [`06_clustering_transfer_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level2/06_clustering_transfer_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- kNN graph + Leiden on the control's Harmony embedding -> my lipizones
- Compare to Allen anatomy (ARI, reciprocal enrichment), marker lipids
- kNN label transfer control -> pregnant with confidence
- Run EUCLID's divisive splitter for real. Save `06_clustered.h5ad`

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB05 output. The thing I cluster on is obsm["X_harmony"]: 12 Harmony-corrected NMF
# programs per pixel. Clustering on 12 de-noised, de-batched numbers is way better (and
# faster) than clustering on 104 raw lipid columns.
adata = ad.read_h5ad(paths.stage("embedded"))

print("pixels x lipids:", adata.shape)
print("embeddings available:", list(adata.obsm))   # expect X_nmf, X_tsne, X_harmony

# EUCLID (the atlas's real clustering engine) was cloned by setup.sh into external/EUCLID.
# I only need it in section 6. When I get there, I put its src/ folder on Python's import
# path so `import EUCLID...` works without pip-installing it:
EUCLID_DIR = paths.REPO_ROOT / "external" / "EUCLID"
print("EUCLID clone found:", EUCLID_DIR.exists())
# import sys; sys.path.insert(0, str(EUCLID_DIR / "src"))

# ---- output -----------------------------------------------------------------------------------
# Adds obs["lipizone"] (my Leiden clusters on control, kNN-transferred to pregnant)
# and obs["euclid_lipizone"] (EUCLID's version, for comparison):
#   adata.write_h5ad(paths.stage("clustered"))   ->  data/derived/06_clustered.h5ad

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time